<a href="https://colab.research.google.com/github/Wor1ock/dsp_labs/blob/main/lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа № 1.1
## *Извлечение признаков из текста*
по курсу Технологии обработки естественного языка  
**направление:** Речевые технологии и машинное обучение  
**преподаватель:** Коротеева Олеся В.  
**выполнил:** Янкин Иван Ю.  
**группа:** М4121

In [1]:
import numpy as np
import pandas as pd
import re
import pymorphy3
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt_tab')
from num2words import num2words
from tqdm.auto import tqdm
tqdm.pandas()
from functools import lru_cache

custom_exceptions = {'нет', 'не', 'опять', 'разве', 'хорошо',
                     'ни', 'ничего',  'наконец',
                     'совсем', 'уже', 'очень', 'чересчур',
                     'слишком', 'никогда'}
stop_words = set(stopwords.words('russian')) - custom_exceptions

import string
punctuations = list(string.punctuation) + ['..', '...', '–', '—', '«', '»', '“', '”']

morph = pymorphy3.MorphAnalyzer()
@lru_cache(maxsize=10000)
def get_morph(word):
    return morph.parse(word)[0]

from pathlib import Path
Path("./features").mkdir(parents=True, exist_ok=True)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vanya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\vanya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
c:\Users\vanya\Documents\1_labs\itmo_master_labs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## загрузка и осмотр датасета

In [2]:
# kaggle datasets download -d maximsuvorov/rutweetcorp -p ./2_sem/nlp/1_lab --unzip

In [3]:
df1 = pd.read_csv('./datasets/negative.csv')
df2 = pd.read_csv('./datasets/positive.csv')

df = pd.concat([df1, df2], ignore_index=True)
assert df.size == df1.size + df2.size
df.head(3)

,id,tdate,tname,ttext,ttype,trep,trtw,tfav,tstcount,tfoll,tfrien,listcount
0,408906762813579264,1386325944,dugarchikbellko,на работе был полный пиддес :| и так каждое за...,-1,0,0,0,8064,111,94,2
1,408906818262687744,1386325957,nugemycejela,"Коллеги сидят рубятся в Urban terror, а я из-з...",-1,0,0,0,26,42,39,0
2,408906858515398656,1386325966,4post21,@elina_4post как говорят обещаного три года жд...,-1,0,0,0,718,49,249,0


In [21]:
tone_df = pd.read_csv('./datasets/words_all_full_rating.csv', sep=';', encoding='cp1251')
tone_dict = dict(zip(tone_df['Words'], tone_df['mean']))
tone_df.head(3)

,Words,mean,dispersion,average rate,Unnamed: 4
0,аборигенный,"-0,25","0,433012701892219",0,NaN
1,аборт,-1,"0,816496580927726",-1,NaN
2,абрамович,0,0,0,NaN


### предобработка текста
- с помощью `nltk.word_tokenize` и `pymorphy3`
- замена чисел, унификация смеха, обработка смешанных слов ('5-го'), токенизация, лемматизация, фильтрация стоп-слов, фильтрация по русскому алфавиту и `-`
- `вход -> выход:` `(text_column : Series[str]) -> (text_column_prepared : Series[list[str]])`

In [5]:
RE_SUFFIX = re.compile(r'\b(\d+)-[а-яё]+\b')
RE_MIXED = re.compile(r'\b\d+[а-яё]+\b')
RE_NUMBER = re.compile(r'\b\d+(?:\.\d+)?')
RE_LAUGH = re.compile(r'\b[авих]{2,}\b')


def text2num(match):
    val = match.group()
    if len(val) > 15: 
        return val
    try:
        return num2words(val, lang='ru')
    except (ValueError, TypeError):
        return val

def split_mixed_numchar(match):
    """
    '22кг' -> 'двадцать два кг'
    """
    word = match.group()
    num = "".join(c for c in word if c.isdigit())
    txt = "".join(c for c in word if c.isalpha())
    return f"{num2words(num, lang='ru')} {txt}".strip()

def clean_patterns(text):
    text = text.lower()
    text = RE_SUFFIX.sub(r'\1', text)
    text = RE_MIXED.sub(split_mixed_numchar, text)
    text = RE_NUMBER.sub(text2num, text)
    text = RE_LAUGH.sub("хах", text)
    return text

def is_valid(token):
    return (
        len(token) > 1
        and all(c in 'абвгдеёжзийклмнопрстуфхцчшщъыьэюя-' for c in token)
        and token not in stop_words
    )

def tokenize_text(text):
    text = text.replace('…', '').replace('–', '-')
    return [t for t in word_tokenize(text, language='russian') if is_valid(t)]

def prepare(text):
    text = clean_patterns(text)
    return [get_morph(t).normal_form for t in tokenize_text(text)]

In [7]:
text_column = df.ttext
text_column_prepared = text_column.progress_apply(prepare)
text_column_prepared.head(3)

100%|██████████| 226834/226834 [01:29<00:00, 2535.55it/s]


0    [работа, полный, пиддес, каждый, закрытие, мес...
1    [коллега, сидеть, рубиться, из-за, долбать, ви...
2                     [говорить, обещаной, год, ждать]
Name: ttext, dtype: object

In [8]:
df.ttext = text_column_prepared
df.to_csv('./datasets/prepared_dataset.csv', index=False)

In [20]:
text_column_prepared.sample(3)

200249                  [создавать, новогодний, настроение]
109492                          [бог, нет, сломать, ноготь]
73794     [мы, привезти, дохуй, конфета, весь, насрать, ...
Name: ttext, dtype: object

### извлечение тоновых признаков
- с помощью `LinisCrowd 2015`
- `вектор сообщения:` среднее, максимальное, минимальное, суммарное значение и количество положительных и отрицательных значений
- `вход -> выход:` `(text_column_prepared : Series[list[str]]) -> (df_tone : DataFrame[float])`

In [10]:
def extract_tone_features(text_tokens, tone_dict=tone_dict):
    tone_scores = [tone_dict[word] for word in text_tokens if word in tone_dict]

    if not tone_scores:
        tone_scores = [0]

    tone_scores = np.array(tone_scores, dtype=float)

    mean = np.mean(tone_scores)
    max_value = np.max(tone_scores)
    min_value = np.min(tone_scores)
    total = np.sum(tone_scores)
    pos_count = len([x for x in tone_scores if x > 0])
    neg_count = len([x for x in tone_scores if x < 0])

    return [mean, max_value, min_value, total, pos_count, neg_count]

In [22]:
text_tone_features = text_column_prepared.progress_apply(extract_tone_features)
df_tone = pd.DataFrame(text_tone_features.tolist(), 
                       columns=['mean', 'max', 'min', 'total', 'pos_count', 'neg_count'])
df_tone.to_csv('./features/tone_features.csv', index=False)
df_tone.head(3)

100%|██████████| 226834/226834 [00:05<00:00, 37907.12it/s]


,mean,max,min,total,pos_count,neg_count
0,0.0,0.0,0.0,0.0,0,0
1,0.0,0.0,0.0,0.0,0,0
2,0.0,0.0,0.0,0.0,0,0


In [23]:
df_tone.sample(5)

,mean,max,min,total,pos_count,neg_count
141860,0.5,1.0,0.0,1.0,1,0
36768,0.0,0.0,0.0,0.0,0,0
29805,0.0,0.0,0.0,0.0,0,0
92897,0.5,1.0,0.0,2.0,2,0
110105,-1.0,-1.0,-1.0,-1.0,0,1


### извлечение морфологических признаков
- с помощью `pymorphy3`
- `вектор сообщения:` для каждого сообщения вычислить относительную частоту основных частей речи
- `вход -> выход:` `(text_column_prepared : Series[list[str]]) -> (morph_df : DataFrame[float])`

In [12]:
from collections import Counter

pos_tags = ['NOUN', 'VERB', 'ADJF', 'ADJS', 'ADVB', 'NUMR', 'PRTF',
            'PRTS', 'NPRO', 'INTJ', 'COMP', 'INFN']

def extract_morph_features(text_tokens):
    if not text_tokens:
        return [0.0] * len(pos_tags)
    
    total_tokens = len(text_tokens)
    text_tags = [get_morph(word).tag.POS for word in text_tokens]
    counter = Counter(text_tags)

    return [counter.get(pos, 0) / total_tokens for pos in pos_tags]

In [13]:
text_morph_features = text_column_prepared.progress_apply(extract_morph_features)
morph_df = pd.DataFrame(text_morph_features.tolist(), 
                        columns=pos_tags)
morph_df.to_csv('./features/morph_features.csv', index=False)
morph_df.head(3)

100%|██████████| 226834/226834 [00:39<00:00, 5794.27it/s]


,NOUN,VERB,ADJF,ADJS,ADVB,NUMR,PRTF,PRTS,NPRO,INTJ,COMP,INFN
0,0.571429,0.0,0.285714,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.142857
1,0.375000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.375000
2,0.250000,0.0,0.250000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.500000


### извлечение tf-idf признаков
- с помощью `TfidfVectorizer`
- `вектор сообщения:` вектор tf-idf
- `вход -> выход:` `(text_column_cleaned : Series[list[str]]) -> (tfidf_df : DataFrame[float])`

In [14]:
text_for_tfidf = text_column_prepared.progress_apply(lambda x: ' '.join(x))

100%|██████████| 226834/226834 [00:00<00:00, 823180.62it/s]


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1000)
tfidf_matrix = vectorizer.fit_transform(text_for_tfidf)

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), 
                        columns=vectorizer.get_feature_names_out())
tfidf_df.to_csv('./features/tfidf_features.csv', index=False)
tfidf_df.head(5)

,автобус,ага,ад,аж,айфон,алгебра,альбом,английский,баба,бабушка,...,штука,шутка,экзамен,эмоция,это,этот,эх,язык,январь,ёлка
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
